# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GourabGorai/FlyRankInternship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

- **Unit of Analysis:** 1 row = 1 unique pseudonymized content asset (`content_id`).
- **Time Window:** Trailing 90-day aggregated performance window, capturing cross-sectional performance across 32 clients.
- **Row Count:** Exactly 30,000 items.

In [1]:
import os, sys, pandas as pd, numpy as np
csv_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(csv_path):
    csv_path = '../../data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(csv_path)
print(f'Grain Check: {len(df):,} total rows, {df["content_id"].nunique():,} unique content_ids across {df["client_id"].nunique()} clients.')
assert len(df) == df['content_id'].nunique(), 'Duplicate unit of analysis found!'


Grain Check: 30,000 total rows, 30,000 unique content_ids across 32 clients.


## 2. Fields: feature / label / context / excluded

Every column in the dataset is classified into one of four mutually exclusive buckets:
1. **Features:** `impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `content_age_days`, `days_since_last_update`, `word_count`, `char_count`, `engagement_rate`, `scroll_rate`, `sessions_90d`, `ai_traffic_pct`, `search_volume`, `competition`, `cpc` (plus engineered log-transforms and ratios).
2. **Label:** `is_declining_label` (`trend_direction == 'down'`).
3. **Context:** `content_id`, `client_id` (used strictly for joining, grouping, and client-holdout validation splits; never as features).
4. **Excluded:** `trend_direction` (the direct source of the label), `trend_pct` (exact percentage change label is derived from — leaks the target 100%), and unverified internal product decision flags.

In [2]:
features = ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update', 'word_count', 'engagement_rate']
label = ['trend_direction']
context = ['content_id', 'client_id']
excluded = ['trend_pct', 'trend_direction']
print(f'Defined Features: {len(features)}')
print(f'Context Columns: {context}')
print(f'Excluded (Leakage) Columns: {excluded}')


Defined Features: 8
Context Columns: ['content_id', 'client_id']
Excluded (Leakage) Columns: ['trend_pct', 'trend_direction']


## 3. Verify it with queries (grain, counts, missing values, windows)

Checking missingness rates and patterns across columns:
- Word count missingness occurs primarily on specific content types (e.g. video/short-form).
- Zero in `avg_position` represents an unranked page (no Google search visibility), not rank 0.

In [3]:
# Query missingness rates
missing = df.isnull().mean()
cols_with_missing = missing[missing > 0].sort_values(ascending=False)
print('Columns with missing values and percentage:')
print(cols_with_missing.round(4))

# Missingness by content_type for word_count
print('\nMissing word_count by content_type:')
print(df.groupby('content_type')['word_count'].apply(lambda x: x.isnull().mean()).round(3))


Columns with missing values and percentage:
provider_used        0.7146
word_count_tier      0.2566
char_count           0.2566
word_count           0.2566
char_count_tier      0.2566
model_used           0.1911
trend_pct            0.1129
competition_level    0.0870
search_volume        0.0823
competition          0.0823
cpc                  0.0823
main_intent          0.0791
scroll_rate          0.0042
dtype: float64

Missing word_count by content_type:
content_type
comparison article    0.000
feedly article        0.000
keyword article       0.283
Name: word_count, dtype: float64


## 4. Data limits

1. **Scale Gotchas:** Rate columns (`ctr`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`) are on a $\times 100$ percentage scale (`0.76` means $0.76\%$).
2. **Special Values:** `avg_position == 0` denotes unranked/unobserved pages (1,205 rows).
3. **Cross-system Ratios:** `scroll_rate` and `ai_traffic_pct` can exceed 100% due to cross-system tracking discrepancies between GSC and GA4.
4. **Cross-Sectional Aggregation:** The data provides a 90-day aggregate snapshot, which does not reflect intra-week keyword seasonality.

In [4]:
unranked_count = (df['avg_position'] == 0).sum()
print(f'Pages with avg_position == 0 (unranked): {unranked_count:,} ({unranked_count/len(df):.2%})')
print(f'CTR max: {df["ctr"].max():.2f}, mean: {df["ctr"].mean():.2f}')
print(f'Scroll rate > 100% rows: {(df["scroll_rate"] > 100).sum():,}')


Pages with avg_position == 0 (unranked): 1,205 (4.02%)
CTR max: 100.00, mean: 0.51
Scroll rate > 100% rows: 119


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.